In [ ]:
import json
import os
import duckdb
# LangChain dependencies
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
# ==============================================================================
# 🔑 AUTENTICACIÓN
# ==============================================================================

import getpass, os
from openai import OpenAI

api_key = getpass.getpass("🔑 Introduce tu OpenAI API Key: ")
client  = OpenAI(api_key=api_key)


In [7]:
pip install langchain langchain-community langchain-openai
pip install faiss-cpu
pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Import the data

In [45]:
# Path to your file
file_path = os.path.join(".", "sql_dataset_bourbaki.json")

# Load the raw data
with open(file_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

## Transform the data into Langchain Documents

In [57]:
extra_storage = {}
docs_for_vector_db = []

for table_name, info in raw_data.items():
    # ID para la tabla
    table_id = f"table_{table_name}"
    docs_for_vector_db.append(Document(page_content=info['description'], metadata={"id": table_id}))

    # Guardamos un dict con el tipo 'schema'
    extra_storage[table_id] = {
        "type": "schema",
        "table_name": table_name,
        "content": info['schema']
    }

    # ID para los ejemplos
    for i, ex in enumerate(info.get('examples', [])):
        ex_id = f"ex_{table_name}_{i}"
        docs_for_vector_db.append(Document(page_content=ex['description'], metadata={"id": ex_id}))

        # Guardamos un dict con el tipo 'sql'
        extra_storage[ex_id] = {
            "type": "sql",
            "content": ex['sql']
        }

## Embeddings model from OpenAI

In [50]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                              openai_api_key = api_key)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o", # or "gpt-3.5-turbo"
    openai_api_key=api_key,
    temperature=0 # Critical for SQL: you want precision, not "creativity"
)

## Create a vectorstore with FAISS

In [58]:
try:
    # Initialize the embeddings
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=api_key
    )

    # Define the vector store
    vectorstore = FAISS.from_documents(docs_for_vector_db, embeddings)

    print("✅ Success! Vector store created.")

except Exception as e:
    print(f"❌ Still getting an error: {e}")

✅ Success! Vector store created.


### View Embeddings from the `embeddings` object

In [53]:
print(f"Content of the first document: {docs_list[0].page_content}")
embedding_example = embeddings.embed_query(docs_list[0].page_content)
print(f"Embedding of the first document (first 10 elements): {embedding_example[:10]}")
print(f"Length of the embedding: {len(embedding_example)}")

Content of the first document: Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.
Embedding of the first document (first 10 elements): [0.0313720703125, 0.0193023681640625, 0.03387451171875, -0.003582000732421875, 0.0247344970703125, 0.00988006591796875, 0.01486968994140625, 0.00832366943359375, 0.002071380615234375, -0.0185089111328125]
Length of the embedding: 1536


### View Embeddings directly from the FAISS index

In [54]:
# FAISS stores vectors as numpy arrays. We can reconstruct them.
# reconstruct_n(index, count) retrieves 'count' vectors starting from 'index'
faiss_vector = vectorstore.index.reconstruct_n(0, 1)
print(f"First vector in FAISS index (first 10 elements): {faiss_vector[0][:10]}")
print(f"Length of the FAISS vector: {len(faiss_vector[0])}")

First vector in FAISS index (first 10 elements): [ 0.03137207  0.01930237  0.03387451 -0.003582    0.0247345   0.00988007
  0.01486969  0.00832367  0.00207138 -0.01850891]
Length of the FAISS vector: 1536


In [82]:
from langchain_core.prompts import ChatPromptTemplate

system_message = """
You are an expert SQL assistant. Use the provided table schemas and examples
to write a valid SQL query based on the user's question.

### DATABASE CONTEXT:
{context}

### INSTRUCTIONS:
- Only return the SQL code.
- Do not explain the code unless asked.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", "{question}")
])

In [86]:
def generate_sql(user_question, verbose = False):
    # 1. Buscamos en VectorDB
    docs = vectorstore.similarity_search(user_question, k=3)

    context_parts = []

    # 2. Hidratamos desde el storage externo
    for doc in docs:
        resource_id = doc.metadata.get("id")
        data = extra_storage.get(resource_id) # Esto devuelve el dict que creamos arriba
        if verbose == True:
          print(doc)
        else:
          continue
        if not data:
            continue

        # Ahora sí, el key "type" existe dentro de 'data'
        if data["type"] == "schema":
            info = f"TABLA: {data['table_name']}\nSCHEMA: {data['content']}"
        elif data["type"] == "sql":
            info = f"EJEMPLO DE CONSULTA: {doc.page_content}\nSQL: {data['content']}"

        context_parts.append(info)

    context_string = "\n\n".join(context_parts)

    # 3. Prompt y LLM
    final_prompt = prompt_template.format(context=context_string, question=user_question)
    if verbose == True:
      print(f"--- PROMPT ENVIADO ---\n{final_prompt}\n----------------------")
    response = llm.invoke(final_prompt)

    return response.content

# --- TEST IT ---
answer = generate_sql("Escribe una consulta para calcular nuestros ingresos totales de hoy y también lista todos los productos que actualmente tienen 0 stock")
print(answer)

```sql
SELECT 
    SUM(o.total_amount) AS total_revenue_today
FROM 
    orders o
WHERE 
    DATE(o.order_date) = CURRENT_DATE;

SELECT 
    p.product_id, p.product_name
FROM 
    products p
WHERE 
    p.stock_quantity = 0;
```


In [89]:
# --- TEST IT ---
answer = generate_sql("De que tabla puedo extraer datos de ordenes?")
print(answer)

```sql
SELECT * FROM information_schema.tables WHERE table_name LIKE '%orden%';
```


# Test the agent with real DB

In [98]:

# 1. Create the in-memory DuckDB connection
con = duckdb.connect(database=':memory:')

# 2. Create tables and insert rows
setup_sql = """
-- USERS
CREATE TABLE users (id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
INSERT INTO users VALUES
(1, 'Ana', 'Gomez', 'cliente@example.com', '555-0101', '2026-02-15 10:00:00'),
(2, 'Carlos', 'Ruiz', 'carlos.r@mail.com', NULL, '2026-03-01 14:30:00'),
(3, 'Beatriz', 'Perez', 'bea.p@mail.com', '555-0103', '2026-03-10 09:15:00'),
(4, 'David', 'Lopes', 'david.l@mail.com', NULL, '2026-01-20 11:20:00'),
(5, 'Elena', 'Torres', 'elena.t@mail.com', '555-0105', '2026-03-16 16:45:00'),
(6, 'Fernando', 'Silva', 'fer.s@mail.com', '555-0106', '2026-02-28 08:00:00'),
(7, 'Gabriela', 'Cruz', 'gabi.c@mail.com', '555-0107', '2026-03-05 13:10:00'),
(8, 'Hugo', 'Diaz', 'hugo.d@mail.com', NULL, '2026-03-12 17:55:00'),
(9, 'Isabel', 'Ortiz', 'isa.o@mail.com', '555-0109', '2026-01-05 12:00:00'),
(10, 'Jorge', 'Rios', 'jorge.r@mail.com', '555-0110', '2026-03-17 09:00:00');

-- CATEGORIES
CREATE TABLE categories (id INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT);
INSERT INTO categories VALUES
(1, NULL, 'Electronics', 'Gadgets and devices'),
(2, NULL, 'Clothing', 'Apparel and accessories'),
(3, 1, 'Laptops', 'Computers and notebooks'),
(4, 1, 'Smartphones', 'Mobile devices'),
(5, 2, 'Menswear', 'Men clothing'),
(6, 5, 'Shirts', 'T-shirts and button-downs'),
(7, 5, 'Pants', 'Jeans and trousers'),
(8, 2, 'Womenswear', 'Women clothing'),
(9, 8, 'Dresses', 'Casual and formal dresses'),
(10, NULL, 'Home & Garden', 'Furniture and tools');

-- PRODUCTS
CREATE TABLE products (id INT PRIMARY KEY, category_id INT, name VARCHAR(255), price DECIMAL(10, 2), stock_quantity INT, sku VARCHAR(50) UNIQUE, is_active BOOLEAN);
INSERT INTO products VALUES
(1, 3, 'Pro Laptop 15', 1299.99, 15, 'PROD-123', TRUE),
(2, 4, 'Smartphone X', 799.50, 0, 'SKU-002', TRUE),
(3, 6, 'Cotton T-Shirt', 19.99, 100, 'SKU-003', TRUE),
(4, 7, 'Denim Jeans', 49.99, 50, 'SKU-004', TRUE),
(5, 9, 'Summer Dress', 39.99, 0, 'SKU-005', FALSE),
(6, 3, 'Basic Laptop 13', 599.00, 30, 'SKU-006', TRUE),
(7, 10, 'Garden Shovel', 25.00, 20, 'SKU-007', TRUE),
(8, 10, 'Table Lamp', 35.50, 10, 'SKU-008', FALSE),
(9, 4, 'Budget Phone', 299.00, 0, 'SKU-009', TRUE),
(10, 6, 'Polo Shirt', 29.99, 45, 'SKU-010', TRUE);

-- ORDERS
CREATE TABLE orders (id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2));
INSERT INTO orders VALUES
(1, 1, '2024-05-10 10:00:00', 'cancelled', 1299.99),
(2, 1, '2026-03-17 08:30:00', 'pending', 49.99),
(3, 2, '2026-03-16 14:00:00', 'completed', 799.50),
(4, 3, '2024-11-20 09:15:00', 'cancelled', 39.99),
(5, 4, '2026-03-17 11:20:00', 'completed', 599.00),
(6, 5, '2026-02-28 16:45:00', 'completed', 19.99),
(7, 6, '2026-03-15 08:00:00', 'pending', 25.00),
(8, 7, '2026-03-17 13:10:00', 'pending', 35.50),
(9, 8, '2024-01-12 17:55:00', 'completed', 299.00),
(10, 9, '2026-03-17 12:00:00', 'pending', 29.99),
(11, 10, '2026-03-18 14:53:00', 'completed', 1099.00);

-- ORDER_ITEMS
CREATE TABLE order_items (id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, unit_price DECIMAL(10, 2));
INSERT INTO order_items VALUES
(1, 1, 1, 1, 1299.99),
(2, 2, 4, 1, 49.99),
(3, 3, 2, 1, 799.50),
(4, 4, 5, 1, 39.99),
(5, 5, 6, 1, 599.00),
(6, 6, 3, 1, 19.99),
(7, 7, 7, 1, 25.00),
(8, 8, 8, 1, 35.50),
(9, 9, 9, 1, 299.00),
(10, 10, 10, 1, 29.99);

-- REVIEWS
CREATE TABLE reviews (id INT PRIMARY KEY, product_id INT, user_id INT, rating INT, comment TEXT, created_at TIMESTAMP);
INSERT INTO reviews VALUES
(1, 1, 2, 5, 'Amazing laptop!', '2026-03-15 10:00:00'),
(2, 2, 3, 1, 'Battery drains too fast.', '2026-02-20 14:30:00'),
(3, 3, 4, 4, 'Good quality cotton.', '2026-03-10 09:15:00'),
(4, 1, 5, 5, 'Best purchase ever.', '2026-03-16 11:20:00'),
(5, 5, 6, 2, 'Size runs small.', '2026-01-20 16:45:00'),
(6, 6, 7, 4, 'Great value for money.', '2026-02-28 08:00:00'),
(7, 7, 8, 5, 'Sturdy and reliable.', '2026-03-05 13:10:00'),
(8, 8, 9, 1, 'Arrived broken.', '2026-03-12 17:55:00'),
(9, 9, 10, 3, 'Its okay for the price.', '2026-01-05 12:00:00'),
(10, 10, 1, 5, 'Fits perfectly.', '2026-03-17 09:00:00');

-- SHIPPING_DETAILS
CREATE TABLE shipping_details (id INT PRIMARY KEY, order_id INT, address_line1 VARCHAR(255), city VARCHAR(100), state VARCHAR(100), postal_code VARCHAR(20), tracking_number VARCHAR(100), carrier VARCHAR(50));
INSERT INTO shipping_details VALUES
(1, 1, '123 Main St', 'Bogota', 'Cundinamarca', '110111', 'TRK-001', 'FedEx'),
(2, 2, '456 Elm St', 'Medellin', 'Antioquia', '050001', 'TRK-002', 'UPS'),
(3, 3, '789 Oak Ave', 'Cali', 'Valle', '760001', 'TRK-003', 'FedEx'),
(4, 4, '321 Pine Rd', 'Cartagena', 'Bolivar', '130001', 'TRK-004', 'DHL'),
(5, 5, '654 Cedar Ln', 'Barranquilla', 'Atlantico', '080001', 'TRK-005', 'FedEx'),
(6, 6, '987 Birch Blvd', 'Bucaramanga', 'Santander', '680001', 'TRK-006', 'UPS'),
(7, 7, '147 Walnut St', 'Pereira', 'Risaralda', '660001', 'TRK-007', 'FedEx'),
(8, 8, '258 Cherry Ct', 'Manizales', 'Caldas', '170001', 'TRK-008', 'DHL'),
(9, 9, '369 Spruce Way', 'Santa Marta', 'Magdalena', '470001', 'TRK-009', 'FedEx'),
(10, 10, '741 Ash Dr', 'Cucuta', 'Norte de Santander', '540001', 'TRK-010', 'UPS');

-- PROMOTIONS
CREATE TABLE promotions (id INT PRIMARY KEY, code VARCHAR(50) UNIQUE, discount_percentage DECIMAL(5, 2), start_date DATE, end_date DATE, is_active BOOLEAN);
INSERT INTO promotions VALUES
(1, 'VERANO20', 20.00, '2026-03-01', '2026-03-31', TRUE),
(2, 'WELCOME10', 10.00, '2026-01-01', '2026-12-31', TRUE),
(3, 'FLASH50', 50.00, '2026-03-15', '2026-03-20', TRUE),
(4, 'WINTER30', 30.00, '2025-12-01', '2026-02-28', FALSE),
(5, 'VIP25', 25.00, '2026-01-01', '2026-12-31', TRUE),
(6, 'SPRING15', 15.00, '2026-03-20', '2026-06-20', FALSE),
(7, 'BF2025', 40.00, '2025-11-25', '2025-11-30', FALSE),
(8, 'CYBER2025', 45.00, '2025-12-01', '2025-12-02', FALSE),
(9, 'FREESHIP', 100.00, '2026-03-01', '2026-03-31', TRUE),
(10, 'HALLOWEEN', 15.00, '2025-10-25', '2025-10-31', FALSE);
"""

# Run the setup SQL
con.execute(setup_sql)
print("Database successfully created and populated with 10 rows per table in DuckDB!")



Database successfully created and populated with 10 rows per table in DuckDB!


Let's try to query the database using a natural language question and the `generate_sql` function, then execute the generated SQL against the DuckDB connection.

In [105]:
# Example user question
user_question = "Escribe una consulta para calcular nuestros ingresos totales de la ultima semana y también lista todos los productos que actualmente tienen 0 stock"

# Generate the SQL query
generated_sql = generate_sql(user_question, verbose = True)

# Execute the generated SQL query using DuckDB
try:
    parsed_sql_generated = generated_sql.replace("```", "").replace("sql", "")
    print(f"Generated SQL:\n{parsed_sql_generated}")
except Exception as e:
    print(f"Error executing SQL query: {e}")

page_content='Listar productos inactivos o sin stock.' metadata={'id': 'ex_products_0'}
page_content='Calcular el valor total del inventario para un producto.' metadata={'id': 'ex_products_2'}
page_content='Calcular los ingresos totales de hoy.' metadata={'id': 'ex_orders_1'}
--- PROMPT ENVIADO ---
System: 
You are an expert SQL assistant. Use the provided table schemas and examples 
to write a valid SQL query based on the user's question.

### DATABASE CONTEXT:
EJEMPLO DE CONSULTA: Listar productos inactivos o sin stock.
SQL: SELECT name, sku FROM products WHERE stock_quantity = 0 OR is_active = FALSE;

EJEMPLO DE CONSULTA: Calcular el valor total del inventario para un producto.
SQL: SELECT name, (price * stock_quantity) AS total_value FROM products WHERE sku = 'PROD-123';

EJEMPLO DE CONSULTA: Calcular los ingresos totales de hoy.
SQL: SELECT SUM(total_amount) AS revenue FROM orders WHERE DATE(order_date) = CURRENT_DATE AND status != 'cancelled';

### INSTRUCTIONS:
- Only return the

In [106]:
result_df = con.execute("""
SELECT SUM(total_amount) AS total_revenue_last_week
FROM orders
WHERE order_date >= CURRENT_DATE - INTERVAL '7 days'
AND status != 'cancelled';
""").df()
print("\nQuery Result:")
display(result_df)


Query Result:


,total_revenue_last_week
0,2637.98
